# IZQAR PaySim v2 sealed final evaluation
This notebook is for exactly one final evaluation of the validation-selected clean `pre_transaction_v2` configuration. It does not tune hyperparameters or the threshold. Add Input: `ealaxi/paysim1`; do not publish source CSV data.


In [ ]:
from pathlib import Path
import subprocess, sys
PINNED_REVISION = '09a7830dbfeeec2d97a52f3b6bf40cb82ef700fe'
root = Path('/kaggle/working/izqar')
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feat/ml-pipelines-and-startup', 'https://github.com/Night011/izqar.git', str(root)], check=True)
actual = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
if actual != PINNED_REVISION:
    raise RuntimeError(f'Pinned source mismatch: expected {PINNED_REVISION}, got {actual}; do not run an unpinned final evaluation.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(root / 'backend/requirements-ml.txt')], check=True)
matches = list(Path('/kaggle/input').rglob('PS_20174392719_1491204439457_log.csv'))
if len(matches) != 1:
    raise FileNotFoundError(f'Expected one PaySim CSV under /kaggle/input; found {len(matches)}. Use Add Input: ealaxi/paysim1.')
print('PaySim input:', matches[0], 'bytes:', matches[0].stat().st_size)


In [ ]:
from pathlib import Path
import subprocess, sys, zipfile
backend = root / 'backend'
output = Path('/kaggle/working/paysim_v5_pre_transaction_v2_sealed.joblib')
command = [sys.executable, str(backend / 'scripts/train_paysim.py'), '--data', str(matches[0]), '--output', str(output), '--estimators', '180', '--full-data', '--mode', 'final', '--feature-set', 'pre_transaction_v2', '--frozen-final-feature-set', 'pre_transaction_v2', '--model-version', 'paysim-xgboost-v5']
subprocess.run(command, cwd=backend, check=True)
report = output.with_suffix('.metrics.json')
if not output.is_file() or not report.is_file():
    raise RuntimeError('Trainer did not produce both final artifact and metrics report.')
bundle = Path('/kaggle/working/izqar_paysim_v5_sealed_artifacts.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(output, output.name)
    archive.write(report, report.name)
print('Sealed artifact output:', bundle)
